
# 📘 06_lakehouse_monitoring_API_v0.2.ipynb
This notebook extends the previous **v0.1** version, which used only one metadata table (`monitors_control`).

In **v0.2**, we introduce two additional metadata tables:
- **metric_templates** — defines reusable metric logic (using SQL functions)
- **metric_bindings** — maps templates to specific monitored tables

Together with `monitors_control`, these tables form a fully metadata-driven monitoring framework where:
1. The **templates** describe *what* to measure.
2. The **bindings** link metrics to tables.
3. The **control** table determines *how* and *when* to monitor.

### Framework Overview

<pre>
┌────────────────────┐
│  SQL Functions     │  ← reusable rule logic (04_Custom_Metric_Functions)
├────────────────────┤
│  Metric Templates  │  ← define metric expressions using SQL functions
├────────────────────┤
│  Metric Bindings   │  ← attach templates to specific tables
├────────────────────┤
│  Monitors Control  │  ← schedule, enable, and manage monitors
└────────────────────┘
</pre>

This notebook reads all three metadata layers and automatically creates or updates Lakehouse Monitors using the Databricks SDK.

In [0]:
dbutils.widgets.text("catalog", "dbdemos_steventan", "Catalog")
dbutils.widgets.text("admin_schema", "monitoring_admin", "Admin Schema")
dbutils.widgets.text("out_schema", "lakehouse_monitoring_demo_results", "Output Schema")
dbutils.widgets.text("assets_dir_base", "/Workspace/Users/steven.tan@databricks.com/", "Assets Dir Base")
dbutils.widgets.text("data_schema", "lakehouse_monitoring", "Data Schema")

catalog = dbutils.widgets.get("catalog")
admin_schema = dbutils.widgets.get("admin_schema")
out_schema = dbutils.widgets.get("out_schema")
assets_dir_base = dbutils.widgets.get("assets_dir_base")
data_schema = dbutils.widgets.get("data_schema")

### Step 1 — Load Metadata

The notebook loads and filters all metadata tables from Unity Catalog.
- `monitors_control` → monitor orchestration  
- `metric_templates` → reusable metric definitions  
- `metric_bindings` → mapping between templates and data tables

Only entries with `enabled = true` will be considered.

In [0]:
from pyspark.sql import functions as F

ctrl_df      = spark.table(f"{catalog}.{admin_schema}.monitors_control").filter("enabled = true")
templates_df = spark.table(f"{catalog}.{admin_schema}.metric_templates")
bindings_df  = spark.table(f"{catalog}.{admin_schema}.metric_bindings").filter("enabled = true")

display(ctrl_df)
display(templates_df.select("template_name","description","metric_type","output_spark_type"))
display(bindings_df.select("table_catalog","table_schema","table_name","metric_name","template_name","enabled"))

### Step 2 — What Changed from v0.1

- Added lookup logic for `metric_templates` and `metric_bindings`.
- Each monitor now automatically attaches **custom metrics** defined via metadata.
- Still idempotent — re-running this notebook will not duplicate monitors.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import (
    MonitorMetric, MonitorMetricType, MonitorTimeSeries,
    MonitorCronSchedule, MonitorNotifications
)
from pyspark.sql import types as T

w = WorkspaceClient()

# Map simple strings to SDK enum
_METRIC_ENUM = {
    "AGGREGATE": MonitorMetricType.CUSTOM_METRIC_TYPE_AGGREGATE,
    "DERIVED":   MonitorMetricType.CUSTOM_METRIC_TYPE_DERIVED,
    "DRIFT":     MonitorMetricType.CUSTOM_METRIC_TYPE_DRIFT,
}

def _spark_type_json_str(simple: str) -> str:
    """Return a JSON-encoded Spark data type field for MonitorMetric.output_data_type."""
    t = (simple or "").strip().lower()
    if t == "double":    dt = T.DoubleType()
    elif t == "long":    dt = T.LongType()
    elif t == "boolean": dt = T.BooleanType()
    else:                dt = T.StringType()
    return T.StructField("output", dt).json()

# Cache templates on driver (name → Row)
_template_map = {r["template_name"]: r for r in templates_df.collect()}

def _render_metric_from_rows(tmpl_row, bind_row) -> MonitorMetric:
    """
    Build a MonitorMetric by replacing {{PARAM}} placeholders in the template definition
    with binding params; pass through input_columns and output type.
    """
    definition = tmpl_row.definition_template
    params = bind_row.params or {}
    for k, v in params.items():
        definition = definition.replace("{{" + k + "}}", v)

    input_cols = bind_row.input_columns
    if isinstance(input_cols, str):
        input_cols = [input_cols]

    return MonitorMetric(
        type=_METRIC_ENUM[tmpl_row.metric_type.upper()],
        name=bind_row.metric_name,
        input_columns=input_cols,
        definition=definition,
        output_data_type=_spark_type_json_str(tmpl_row.output_spark_type)
    )

def _metrics_for_table(tc: str, ts: str, tn: str):
    """
    Look up bindings for a table and render the full MonitorMetric list.
    """
    b = bindings_df.filter(
        (F.col("table_catalog")==tc) &
        (F.col("table_schema")==ts) &
        (F.col("table_name")==tn)
    ).collect()

    if not b:
        return []

    metrics = []
    for row in b:
        tmpl = _template_map.get(row.template_name)
        if not tmpl:
            # template missing — skip this binding
            continue
        metrics.append(_render_metric_from_rows(tmpl, row))
    return metrics

def _safe_refresh(table_fqn: str):
    """
    Trigger a refresh only if the monitor isn't in a pending state.
    """
    try:
        mon = w.quality_monitors.get(table_name=table_fqn)
        # If status is PENDING, skip to avoid API error; otherwise refresh
        status = getattr(mon, "status", None)
        if getattr(status, "state", None) == "MONITOR_STATUS_PENDING":
            return "skipped (pending)"
        w.quality_monitors.run_refresh(table_name=table_fqn)
        return "queued"
    except Exception as e:
        # If we cannot fetch, do nothing
        return f"skip ({type(e).__name__})"

In [0]:
results = []

for r in ctrl_df.collect():
    table_fqn = f"{r.table_catalog}.{r.table_schema}.{r.table_name}"

    # Render custom metrics from metadata (may be empty)
    custom_metrics = _metrics_for_table(r.table_catalog, r.table_schema, r.table_name)

    # Desired monitor spec
    desired = dict(
        table_name=table_fqn,
        output_schema_name=r.output_schema_name,
        custom_metrics=custom_metrics
    )

    # Time-series config
    if r.timestamp_col:
        desired["time_series"] = MonitorTimeSeries(
            timestamp_col=r.timestamp_col,
            granularities=list(r.granularities) if r.granularities else ["1 day"]
        )

    # Optional schedule
    if r.schedule_cron and r.schedule_tz:
        desired["schedule"] = MonitorCronSchedule(
            quartz_cron_expression=r.schedule_cron,
            timezone_id=r.schedule_tz
        )

    # Optional notifications
    if r.notifications_on_failure:
        desired["notifications"] = MonitorNotifications(
            on_failure={"email_addresses": list(r.notifications_on_failure)}
        )

    # Create/update
    try:
        existing = w.quality_monitors.get(table_name=table_fqn)
        # Update existing monitor (no assets_dir on update)
        w.quality_monitors.update(**desired)
        refresh_note = _safe_refresh(table_fqn)
        results.append((table_fqn, "updated", f"refresh {refresh_note}"))
    except Exception:
        # Create new monitor (assets_dir required on create)
        w.quality_monitors.create(assets_dir=r.assets_dir, **desired)
        refresh_note = _safe_refresh(table_fqn)
        results.append((table_fqn, "created", f"refresh {refresh_note}"))

for t, a, n in results:
    print(f"• {t} -> {a} | {n}")

### Summary

This v0.2 runner:

- Loads all three metadata tables (`monitors_control`, `metric_templates`, `metric_bindings`)
- Renders **custom metrics** per table using template definitions and binding parameters
- Creates or updates monitors idempotently
- Triggers a safe refresh (skips when the monitor is pending)

Next steps:
- Add/modify rows in `metric_templates` and `metric_bindings` to evolve your rules without changing code
- Use dashboards to visualize results and iterate on rules